# ORC Basics — 02: Writing ORC Files

## What you will learn
ORC write options control stripe size, compression, bloom filters,
and row index stride — each affecting read performance differently.

In this notebook:
1. Basic write and save modes
2. Compression codecs — zlib vs snappy vs lz4 vs zstd vs none
3. Stripe size — the ORC equivalent of Parquet row groups
4. Row index stride — granularity of row-level skipping
5. Bloom filters — fast equality lookups
6. Writing sorted data for maximum statistics benefit


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/orc_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('orc-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price,2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows")

## Step 1 — Basic Write

In [ ]:
OUT = f"{DATA_DIR}/write_test"
df.write.mode("overwrite").orc(f"{OUT}/default")
files = glib.glob(f"{OUT}/default/*.orc")
mb = sum(pathlib.Path(f).stat().st_size for f in files)/1024/1024
print(f"Default write: {len(files)} file(s), {mb:.1f} MB")

## Step 2 — Compression Codecs

In [ ]:
codecs = ["none","snappy","zlib","lz4","zstd"]
results = {}
for codec in codecs:
    path = f"{OUT}/codec_{codec}"
    t0=time.time(); df.write.mode("overwrite").option("compression",codec).orc(path); t_w=time.time()-t0
    files=glib.glob(f"{path}/*.orc")
    mb=sum(pathlib.Path(f).stat().st_size for f in files)/1024/1024
    t0=time.time(); spark.read.orc(path).agg(F.sum("revenue")).collect(); t_r=time.time()-t0
    results[codec]={"write":t_w,"size":mb,"read":t_r}

base=results["none"]["size"]
print(f"{'Codec':<8} {'Write':>8} {'MB':>8} {'vs none':>8} {'Read':>8}")
print("-"*46)
for c,r in results.items():
    print(f"  {c:<6} {r['write']:>6.2f}s {r['size']:>6.1f} MB {r['size']/base:>7.2f}x {r['read']:>6.3f}s")
print()
print("zlib: highest compression (default in ORC, equivalent to gzip)")
print("snappy: balanced (recommended for hot data)")
print("zstd: modern high-ratio codec")

## Step 3 — Stripe Size: ORC's Row Group

In [ ]:
# Stripe size controls granularity of data skipping
# Smaller stripes → more precise skipping but more overhead
stripe_sizes = [("8mb",8), ("64mb",64), ("256mb",256)]
print(f"{'Config':<10} {'MB':>8} {'Files':>8} {'Full scan':>12} {'Filter scan':>14}")
print("-"*58)
for label, mb_val in stripe_sizes:
    path = f"{OUT}/stripe_{label}"
    df.write.mode("overwrite") \
            .option("orc.stripe.size", str(mb_val*1024*1024)) \
            .option("compression","snappy").orc(path)
    files = glib.glob(f"{path}/*.orc")
    size_mb = sum(pathlib.Path(f).stat().st_size for f in files)/1024/1024
    tf, ts = [], []
    for _ in range(3):
        t0=time.time(); spark.read.orc(path).agg(F.sum("revenue")).collect(); tf.append(time.time()-t0)
        t0=time.time(); spark.read.orc(path).filter(F.col("category")=="Electronics").count(); ts.append(time.time()-t0)
    print(f"  {label:<8} {size_mb:>6.1f} MB {len(files):>6} {sorted(tf)[1]:>10.3f}s {sorted(ts)[1]:>12.3f}s")

## Step 4 — Bloom Filters

In [ ]:
# Bloom filters pre-filter rows for equality checks
# Especially useful for high-cardinality string columns
BF_PATH = f"{OUT}/with_bloom"
df.write.mode("overwrite") \
        .option("orc.bloom.filter.columns", "category,status,country") \
        .option("orc.bloom.filter.fpp", "0.05") \
        .orc(BF_PATH)

NO_BF_PATH = f"{OUT}/no_bloom"
df.write.mode("overwrite").orc(NO_BF_PATH)

# Benchmark equality filter
tbf, tnbf = [], []
for _ in range(3):
    t0=time.time(); spark.read.orc(BF_PATH).filter((F.col("category")=="Electronics")&(F.col("status")=="confirmed")).count(); tbf.append(time.time()-t0)
    t0=time.time(); spark.read.orc(NO_BF_PATH).filter((F.col("category")=="Electronics")&(F.col("status")=="confirmed")).count(); tnbf.append(time.time()-t0)

print(f"WITH bloom filters   : {sorted(tbf)[1]:.3f}s")
print(f"WITHOUT bloom filters: {sorted(tnbf)[1]:.3f}s")
print(f"Bloom filter benefit : {sorted(tnbf)[1]/sorted(tbf)[1]:.1f}x  (more visible with larger files)")
print()
print("fpp = false positive probability (0.05 = 5% false positives)")
print("Lower fpp = more accurate but larger bloom filter size")

## Step 5 — Writing Sorted Data

In [ ]:
# Sort before write → min/max stats become non-overlapping → better skipping
sorted_path   = f"{OUT}/sorted"
unsorted_path = f"{OUT}/unsorted"

df.orderBy("category","country","revenue").write.mode("overwrite").orc(sorted_path)
df.write.mode("overwrite").orc(unsorted_path)

ts, tu = [], []
q = lambda p: spark.read.orc(p).filter((F.col("category")=="Electronics")&(F.col("country")=="US")).agg(F.sum("revenue")).collect()
for _ in range(3):
    t0=time.time(); q(sorted_path);   ts.append(time.time()-t0)
    t0=time.time(); q(unsorted_path); tu.append(time.time()-t0)

print(f"Sorted   (WHERE cat=Electronics AND country=US): {sorted(ts)[1]:.3f}s")
print(f"Unsorted                                       : {sorted(tu)[1]:.3f}s")
print(f"Speedup from sorting: {sorted(tu)[1]/sorted(ts)[1]:.1f}x")

## Summary

```python
df.write.mode('overwrite')
  .option('compression',           'snappy')   # none|snappy|zlib|lz4|zstd
  .option('orc.stripe.size',        str(64*1024*1024))   # 64 MB default
  .option('orc.row.index.stride',   '10000')   # row index every 10K rows
  .option('orc.bloom.filter.columns','cat,status')  # equality filter columns
  .option('orc.bloom.filter.fpp',   '0.05')
  .orc('/path/')
```